### Table Analysis

In [ ]:
import pandas as pd
from src.config import *
import re
import numpy as np
from datetime import datetime

In [ ]:
data=pd.read_csv(DIR_WEIGHTED_DATA)
survey = pd.read_excel(DIR_META_DATA, sheet_name='survey')
statement_allocation=pd.read_csv(DIR_STATEMENT_LINKS)
choices = pd.read_excel(DIR_META_DATA, sheet_name='choices')

In [ ]:
data["drivers"]=data["reason_use_mc"].fillna(data["drivers_nonusers"])
data["main_drivers"]=data["reason_use_main"]

data["barriers"]=data["reason_nonuse_mc"].fillna(data["barriers_users"])
data["main_barriers"]=data["reason_nonuse_main"]

# Geographic split: one column per GEO_SPLIT_COL value (departments for Togo -- the sampling
# strata, taken from the frame by geom_id in 02b, so the labels are consistent).
data["geo"] = data[GEO_SPLIT_COL]
n_missing = data["geo"].isna().sum()
if n_missing:
    print(f"WARNING: {n_missing} respondents with no {GEO_SPLIT_COL} (excluded from the {GEO_SPLIT_LABEL} split)")
print(f"{GEO_SPLIT_LABEL} split groups: {data['geo'].value_counts().to_dict()}")

User input to adjust the user categories to group all non users (pastusers + future users):

In [ ]:
group = input("Group non-users (1 - yes / 0 - no)")

In [ ]:
# initialise variables for output file name
today_date = datetime.today()
date_string = today_date.strftime("%y_%m_%d")

# adjust groupings based on user input
if group == "1":
    data.loc[data['use'].isin(["future_user", "past_user"]), 'use'] = 'nonuser'
    USER_CATEGORIES = ["user", "nonuser"]
    out_path = Path(str(DIR_DRIVERS_BARRIERS_TABLE_GROUPED).format(date=date_string))

else:
    USER_CATEGORIES = ["future_user", "past_user", "user", "nonuser"]
    out_path = Path(str(DIR_DRIVERS_BARRIERS_TABLE).format(date=date_string))

In [ ]:
TABLE_COLUMNS = [
    'Name',
    'Prevalence (All)', 'N (All)', 'Weighted N (All)',
    'Prevalence (Most important driver/barrier)',
    'Statements', 'Priority', 'Driver/Barrier',
    'AGE_GROUP: Prevalence (All)', 'AGE_GROUP: N (All)', 'AGE_GROUP: Weighted N (All)',
    'AGE_GROUP: Prevalence (Most important driver/barrier)', 'AGE_GROUP: Statements', 'AGE Focus',
    'GENDER: Prevalence (All)', 'GENDER: N (All)', 'GENDER: Weighted N (All)',
    'GENDER: Prevalence (Most important driver/barrier)', 'GENDER: Statements', 'GENDER Focus',
    'URBAN_RURAL: Prevalence (All)', 'URBAN_RURAL: N (All)', 'URBAN_RURAL: Weighted N (All)',
    'URBAN_RURAL: Prevalence (Most important driver/barrier)', 'URBAN_RURAL: Statements', 'URBAN_RURAL Focus',
    f'{GEO_SPLIT_LABEL}: Prevalence (All)', f'{GEO_SPLIT_LABEL}: N (All)', f'{GEO_SPLIT_LABEL}: Weighted N (All)',
    f'{GEO_SPLIT_LABEL}: Prevalence (Most important driver/barrier)', f'{GEO_SPLIT_LABEL}: Statements', f'{GEO_SPLIT_LABEL} Focus',
]
statement_cols = [col for col in data.columns if col.startswith("statement")]

In [ ]:

#TODO: move to cleaning sheet
# map_likert (src/config.py) matches on the French text, so punctuation/Ewe differences don't break it
data[statement_cols] = map_likert(data[statement_cols])


In [ ]:
def format_new_col(df):
    df=df.fillna(0)
    cols=df.columns.tolist()
    df=df.apply(lambda col: str(col.name)+": "+col.astype(str) + '%' )
    df['Combined_Column'] = df.apply(lambda row: ' \n '.join(row), axis=1)
    df.drop(cols,axis=1,inplace=True)
    return df

def value_counts_multiple_answers_weighted(var: pd.Series, weights: pd.Series) -> pd.Series:
    var_name = var.name
    df = pd.DataFrame({'answer': var, 'weight': weights})
    df = df.dropna(subset=['answer'])          
    df['answer'] = df['answer'].astype(str).str.split('|')
    df = df.explode('answer', ignore_index=True) 
    out = df.groupby('answer')['weight'].sum().sort_values(ascending=False)
    out.rename({"answer":var_name})
    out.index.name=var_name
    return out

def split_dribar_weighted(df, split_var,question):
  if df[question].nunique()==0:
     return pd.DataFrame()
  if split_var!="No Split" and split_var!="":
    ret=df.groupby([split_var]).apply(lambda x: value_counts_multiple_answers_weighted(x[question],x["normalized_weight"])).reset_index().rename(columns={"weight":"Count"})
    if ret.shape[0]==1:
          var=ret[split_var].values[0]
          st.warning(f"The selected split is not possible, all interviews are {split_var} ="+var)
          return pd.DataFrame()
    ret=pd.pivot(ret,columns=split_var,index=question,values="Count")#.fillna(0)
    sums=pd.DataFrame(df.groupby(split_var).sum())["normalized_weight"]
    return round(ret.div(sums,axis=1)*100,2)
  
  ret=round(100*value_counts_multiple_answers_weighted(df[question],df["normalized_weight"])/df["normalized_weight"].sum(),2).rename({"weight":"Count"})
  return pd.DataFrame(ret)["weight"]

def no_prev(group: list) -> str:
    """
    function that returns a string with  0% vlaues for all elements given a group
    """
    group =[str(el)+": 0.0%" for el in sorted(group)]
    return " \n ".join(group)

def get_drivers_barriers(data,statement_cols,var):
  drivers_barriers=data[statement_cols]
  if var!="No Split" and var!="":
    var_col=data[var]
    all=pd.concat([var_col,drivers_barriers],axis=1)
    all=all.groupby(var).mean()
  else:
    all=pd.DataFrame(drivers_barriers.mean(axis=0)).rename(columns={0:"All"}).sort_values("All",ascending=False)
    return pd.DataFrame(round(all*100,2))
  temp=pd.DataFrame(round(all*100,2))
  return temp.T

def get_statement_prevalence(data: pd.DataFrame,reason_value: str,category:str,split:str)-> str:
    filtered_statements=statement_allocation.loc[(statement_allocation.label==reason_value) &(statement_allocation.list_name==category),:]
    filtered_statements=filtered_statements.dropna(subset=["linked_statements"])
    if filtered_statements.shape[0]==0:
        return ""    
    if len(filtered_statements)>1:
         print("Error: more than one row found for: ",reason_value," - ",category)
    linked_statments=filtered_statements.linked_statements.iloc[0].split(", ")
    linked_statments=[statement for statement in linked_statments if statement!=""]
   
    #process statements
    ret=""
    if len(linked_statments)>0:
        ret=get_drivers_barriers(data, linked_statments,split)
        ret=ret.fillna(0)
        ret=format_st_col(ret)
    return ret

def get_statement_name(statement_value):
    # BUGFIX (2026-08-24): Benin's questionnaire leaves the survey sheet's `hint`
    # column blank for every statement_* row (unlike Niger's, where `hint` held an
    # English gloss) -- reading `hint` here silently returned NaN for every single
    # statement, which propagated into format_st_col() as `NaN + str -> NaN`,
    # wiping out the entire "Statements" cell even when a link was found. Fall back
    # to the survey `label` (French/Fon bilingual, same text as statement_labels.csv's
    # label_fr), keeping only the French half for readability.
    row = survey[survey.name == statement_value]
    if row.empty:
        return statement_value
    hint = row.hint.iloc[0]
    if pd.notna(hint) and str(hint).strip():
        return hint
    label = str(row.label.iloc[0])
    return label.split("/", 1)[0].strip() if "/" in label else label.strip()

def format_st_col(df):
    ### function that takes a df grouped agreement percentages and returns a str containing questions and answers in a string
    cols=df.columns.tolist()
    df=df.fillna(0)
    df=df.apply(lambda col: str(col.name)+": "+col.astype(str) + '%' )
    df['Combined_Column'] = df.apply(lambda row: ' \n '.join(row), axis=1)
    df.drop(cols,axis=1,inplace=True)
    df=df.reset_index()
    df["index"]= df["index"].apply(get_statement_name)
    df['Combined_Column'] = df["index"]+' \n '+df["Combined_Column"]
    df.drop(["index"],inplace=True,axis=1)
    merged_str="\n\n".join(df.Combined_Column)+"\n\n"
    return merged_str



def get_top(st,data,category):
    val_count_most=value_counts_multiple_answers_weighted(data[f"main_{category}"], data["normalized_weight"]).reset_index()
    val_count=value_counts_multiple_answers_weighted(data[category], data["normalized_weight"]).reset_index()
    if st in val_count[category].tolist():
        top=val_count[val_count[category] == st].index[0]+1
    else:
        top=100
    if st in val_count_most[f"main_{category}"].tolist():
        top_most=val_count_most[val_count_most[f"main_{category}"] == st].index[0]+1
    else:
        top_most=100
    return min([top,top_most])


def get_state_diff(state):
    state=str(state)
    if state=="" or pd.isna(state):
        return 0
    user=re.findall(string=state,pattern="\\n user: (\d{1,3}.\d{1,2})\%")
    nonuser=re.findall(string=state,pattern="nonuser: (\d{1,3}.\d{1,2})\%")
    if len(user)==0 and len(nonuser)==0:
        # if statment prevalence not found its a question, so not relevant for this
        return 0
    user = [float(x) for x in user]
    nonuser = [float(x) for x in nonuser]
    result = list(map(lambda a, b: round(abs(a - b),2), user, nonuser))
    return sum(result)/len(result)

def get_prio(row,category,data):
    state_diff=get_state_diff(row["Statements"])
    top=get_top(row["Name"],data,category)
    #Very high
    if top <=3 or state_diff>=20 or (top<=5 and state_diff>=10):
        return "Very high"
    if top <=5 or state_diff>=15 or (top<=7 and state_diff>=10):
        return "High"
    if top <=7 or state_diff>=10 or (top<=7 and state_diff>=7):
        return "Medium"

    return "Low"

def _split_prefix(var):
    # Table columns are named "<PREFIX>: ...". For the geographic split the data column is
    # "geo" but the columns use GEO_SPLIT_LABEL (e.g. "DEPARTMENT: Prevalence (All)").
    return GEO_SPLIT_LABEL if var == "geo" else var.upper()

def focus_group(merged_row,data, var, threshold):
    all_ele=data[var].dropna().unique().tolist()
    focus_pre=above_avg(merged_row,data, var, threshold,"Prevalence (All)")
    focus_prev_most=above_avg(merged_row,data, var, threshold,"Prevalence (Most important driver/barrier)")
    focus_state=above_avg(merged_row,data, var, threshold,"Statements")
    all_focus=list(set(focus_pre+focus_prev_most+focus_state))
    if len(all_focus)==0:
        return sorted(all_ele)
    return sorted(all_focus)

def above_avg(merged_row,data, var, threshold,col):
    state=merged_row[f"{_split_prefix(var)}: {col}"]
    # dropna: respondents with no value for the split (e.g. no department) would add NaN here
    unique_ele=data[var].dropna().unique().tolist()
    if pd.isna(state)|(state==""):
        return []
    df=pd.DataFrame(columns=unique_ele)
    for n in df.columns.tolist():
        pattern=re.escape(str(n))+r": (\d{1,3}.\d{1,2})\%"
        found=re.findall(string=state,pattern=pattern)
        if len(found)>0:            
            df.loc[0,n]=float(found[0])
    df=df.fillna(0)
    avg=df.mean(axis=1).mean()
    df=df[df > avg*threshold].dropna(axis=1)
    focus_cat=df.columns.tolist()
    return focus_cat


In [ ]:
# ── Count helpers ──────────────────────────────────────────────────────────────

def value_counts_multiple_answers_unweighted(var: pd.Series) -> pd.Series:
    """Raw respondent count for each answer in a multi-select variable.
    Returns a Series with the same shape/naming as value_counts_multiple_answers_weighted
    so the two are interchangeable in split_dribar_* functions."""
    var_name = var.name
    df = pd.DataFrame({'answer': var})
    df = df.dropna(subset=['answer'])
    df['answer'] = df['answer'].astype(str).str.split('|')
    df = df.explode('answer', ignore_index=True)
    # Use a weight of 1 per row — same structure as weighted version
    df['weight'] = 1.0
    out = df.groupby('answer')['weight'].sum().sort_values(ascending=False)
    out.index.name = var_name
    return out  # name is "weight", same as weighted version → pivot works identically


def split_dribar_unweighted(df, split_var, question):
    """Unweighted percentage prevalence — mirrors split_dribar_weighted but with raw counts.
    Denominator is number of respondents per group (not sum of weights)."""
    if df[question].nunique() == 0:
        return pd.DataFrame()
    if split_var not in ("No Split", ""):
        ret = (
            df.groupby([split_var])
            .apply(lambda x: value_counts_multiple_answers_unweighted(x[question]))
            .reset_index()
            .rename(columns={"weight": "Count"})
        )
        if ret.shape[0] <= 1:
            return pd.DataFrame()
        ret = pd.pivot(ret, columns=split_var, index=question, values="Count")
        sizes = df.groupby(split_var).size()
        return round(ret.div(sizes, axis=1) * 100, 2)
    ret = value_counts_multiple_answers_unweighted(df[question])
    return round(100 * ret / len(df), 2)


def split_dribar_raw_counts(df, split_var, question):
    """Absolute unweighted counts (number of respondents) — not percentages."""
    if df[question].nunique() == 0:
        return pd.DataFrame()
    if split_var not in ("No Split", ""):
        ret = (
            df.groupby([split_var])
            .apply(lambda x: value_counts_multiple_answers_unweighted(x[question]))
            .reset_index()
            .rename(columns={"weight": "Count"})
        )
        if ret.shape[0] <= 1:
            return pd.DataFrame()
        ret = pd.pivot(ret, columns=split_var, index=question, values="Count")
        return ret.fillna(0).astype(int)
    ret = value_counts_multiple_answers_unweighted(df[question])
    return pd.DataFrame(ret)


def split_dribar_weighted_counts(df, split_var, question):
    """Absolute weighted counts (sum of normalized_weight) — not percentages."""
    if df[question].nunique() == 0:
        return pd.DataFrame()
    if split_var not in ("No Split", ""):
        ret = (
            df.groupby([split_var])
            .apply(lambda x: value_counts_multiple_answers_weighted(x[question], x["normalized_weight"]))
            .reset_index()
            .rename(columns={"weight": "Count"})
        )
        if ret.shape[0] <= 1:
            return pd.DataFrame()
        ret = pd.pivot(ret, columns=split_var, index=question, values="Count")
        return round(ret.fillna(0), 1)
    ret = value_counts_multiple_answers_weighted(df[question], df["normalized_weight"])
    return pd.DataFrame(ret)


def no_count(group: list, weighted=False) -> str:
    """Zero-fill string for count columns (no % suffix)."""
    val = "0.0" if weighted else "0"
    group = [str(el) + ": " + val for el in sorted(group)]
    return " \n ".join(group)


def format_count_col(df, weighted=False):
    """Same layout as format_new_col but for counts — no % suffix."""
    df = df.fillna(0)
    cols = df.columns.tolist()
    if weighted:
        df = df.apply(lambda col: str(col.name) + ": " + col.round(1).astype(str))
    else:
        df = df.apply(lambda col: str(col.name) + ": " + col.round(0).astype(int).astype(str))
    df['Combined_Column'] = df.apply(lambda row: ' \n '.join(row), axis=1)
    df.drop(cols, axis=1, inplace=True)
    return df

In [ ]:
def _add_count_cols(st_df, data, split_var, question, n_col, wn_col, fill_group_col):
    """Compute and assign N and Weighted N columns into st_df (in-place)."""
    fill_groups = sorted(data[fill_group_col].dropna().unique().tolist())

    # Raw counts
    df_n = split_dribar_raw_counts(data, split_var, question)
    if not df_n.empty:
        df_n = format_count_col(df_n, weighted=False).reset_index(drop=False)
        st_df.loc[df_n[question].tolist(), n_col] = df_n[["Combined_Column"]].values
    st_df[n_col] = st_df[n_col].fillna(no_count(fill_groups, weighted=False))

    # Weighted counts
    df_wn = split_dribar_weighted_counts(data, split_var, question)
    if not df_wn.empty:
        df_wn = format_count_col(df_wn, weighted=True).reset_index(drop=False)
        st_df.loc[df_wn[question].tolist(), wn_col] = df_wn[["Combined_Column"]].values
    st_df[wn_col] = st_df[wn_col].fillna(no_count(fill_groups, weighted=True))


def create_table(data, category):
    st_df = pd.DataFrame(columns=TABLE_COLUMNS)

    # ── COLUMN 1: Prevalence (All) — unweighted %, split by user category ─────
    df = split_dribar_unweighted(data, "use", category)
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df[["Name", "Prevalence (All)"]] = df[[category, "Combined_Column"]]
    st_df = st_df.set_index("Name")

    not_pres = [col for col in choices[choices["list_name"] == category].label.tolist()
                if col not in st_df.index.tolist()]
    for n_p in not_pres:
        st_df.loc[n_p, "Prevalence (All)"] = no_prev(data["use"].value_counts().keys().tolist())

    # N (All) and Weighted N (All) — split by user category
    _add_count_cols(st_df, data, "use", category, "N (All)", "Weighted N (All)", "use")

    # ── COLUMN 2: Prevalence (Most important) — unweighted % ──────────────────
    df = split_dribar_unweighted(data, "use", f"main_{category}")
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    new_category = [new for new in df[f"main_{category}"] if new not in list(st_df.index)]
    for new_cat in new_category:
        st_df.loc[new_cat] = np.nan
    st_df["Prevalence (All)"].fillna(no_prev(data["use"].value_counts().keys().tolist()), inplace=True)
    st_df.loc[df[f"main_{category}"].tolist(), "Prevalence (Most important driver/barrier)"] = df[["Combined_Column"]].values
    st_df["Prevalence (Most important driver/barrier)"].fillna(
        no_prev(data["use"].value_counts().keys().tolist()), inplace=True)

    # ── AGE_GROUP: Prevalence (All) — unweighted % ────────────────────────────
    df = split_dribar_unweighted(data, "age_group", category)
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[category].tolist(), "AGE_GROUP: Prevalence (All)"] = df[["Combined_Column"]].values
    st_df["AGE_GROUP: Prevalence (All)"].fillna(
        no_prev(data["age_group"].value_counts().keys().tolist()), inplace=True)

    # AGE_GROUP N and Weighted N
    _add_count_cols(st_df, data, "age_group", category,
                    "AGE_GROUP: N (All)", "AGE_GROUP: Weighted N (All)", "age_group")

    # AGE_GROUP: Prevalence (Most important) — unweighted %
    df = split_dribar_unweighted(data, "age_group", f"main_{category}")
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[f"main_{category}"].tolist(), "AGE_GROUP: Prevalence (Most important driver/barrier)"] = df[["Combined_Column"]].values
    st_df["AGE_GROUP: Prevalence (Most important driver/barrier)"].fillna(
        no_prev(data["age_group"].value_counts().keys().tolist()), inplace=True)

    # ── GENDER: Prevalence (All) — unweighted % ───────────────────────────────
    df = split_dribar_unweighted(data, "gender", category)
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[category].tolist(), "GENDER: Prevalence (All)"] = df[["Combined_Column"]].values
    st_df["GENDER: Prevalence (All)"].fillna(
        no_prev(data["gender"].value_counts().keys().tolist()), inplace=True)

    # GENDER N and Weighted N
    _add_count_cols(st_df, data, "gender", category,
                    "GENDER: N (All)", "GENDER: Weighted N (All)", "gender")

    # GENDER: Prevalence (Most important) — unweighted %
    df = split_dribar_unweighted(data, "gender", f"main_{category}")
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[f"main_{category}"].tolist(), "GENDER: Prevalence (Most important driver/barrier)"] = df[["Combined_Column"]].values
    st_df["GENDER: Prevalence (Most important driver/barrier)"].fillna(
        no_prev(data["gender"].value_counts().keys().tolist()), inplace=True)

    # ── URBAN_RURAL: Prevalence (All) — unweighted % ──────────────────────────
    df = split_dribar_unweighted(data, "urban_rural", category)
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[category].tolist(), "URBAN_RURAL: Prevalence (All)"] = df[["Combined_Column"]].values
    st_df["URBAN_RURAL: Prevalence (All)"].fillna(
        no_prev(data["urban_rural"].value_counts().keys().tolist()), inplace=True)

    # URBAN_RURAL N and Weighted N
    _add_count_cols(st_df, data, "urban_rural", category,
                    "URBAN_RURAL: N (All)", "URBAN_RURAL: Weighted N (All)", "urban_rural")

    # URBAN_RURAL: Prevalence (Most important) — unweighted %
    df = split_dribar_unweighted(data, "urban_rural", f"main_{category}")
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[f"main_{category}"].tolist(), "URBAN_RURAL: Prevalence (Most important driver/barrier)"] = df[["Combined_Column"]].values
    st_df["URBAN_RURAL: Prevalence (Most important driver/barrier)"].fillna(
        no_prev(data["urban_rural"].value_counts().keys().tolist()), inplace=True)

    # ── GEO SPLIT (departments): Prevalence (All) — unweighted % ──────────────
    # geo = GEO_SPLIT_COL (see cell above). Respondents with no value are simply
    # absent from the groupby, same as any other split.
    G = GEO_SPLIT_LABEL
    df = split_dribar_unweighted(data, "geo", category)
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[category].tolist(), f"{G}: Prevalence (All)"] = df[["Combined_Column"]].values
    st_df[f"{G}: Prevalence (All)"].fillna(
        no_prev(data["geo"].dropna().value_counts().keys().tolist()), inplace=True)

    # GEO SPLIT N and Weighted N
    _add_count_cols(st_df, data, "geo", category,
                    f"{G}: N (All)", f"{G}: Weighted N (All)", "geo")

    # GEO SPLIT: Prevalence (Most important) — unweighted %
    df = split_dribar_unweighted(data, "geo", f"main_{category}")
    df = format_new_col(df)
    df = df.reset_index(drop=False)
    st_df.loc[df[f"main_{category}"].tolist(), f"{G}: Prevalence (Most important driver/barrier)"] = df[["Combined_Column"]].values
    st_df[f"{G}: Prevalence (Most important driver/barrier)"].fillna(
        no_prev(data["geo"].dropna().value_counts().keys().tolist()), inplace=True)

    # ── STATEMENT COLUMNS ─────────────────────────────────────────────────────
    st_df.reset_index(inplace=True)
    st_df["Statements"] = st_df["Name"].apply(lambda reason: get_statement_prevalence(data, reason, category, "use"))
    st_df["AGE_GROUP: Statements"] = st_df["Name"].apply(lambda reason: get_statement_prevalence(data, reason, category, "age_group"))
    st_df["GENDER: Statements"] = st_df["Name"].apply(lambda reason: get_statement_prevalence(data, reason, category, "gender"))
    st_df["URBAN_RURAL: Statements"] = st_df["Name"].apply(lambda reason: get_statement_prevalence(data, reason, category, "urban_rural"))
    st_df[f"{G}: Statements"] = st_df["Name"].apply(lambda reason: get_statement_prevalence(data, reason, category, "geo"))

    # ── PRIORITY & FOCUS ──────────────────────────────────────────────────────
    st_df["Priority"] = st_df.apply(get_prio, args=[category, data], axis=1)
    st_df["AGE Focus"] = st_df.apply(focus_group, args=[data, "age_group", 1.3], axis=1)
    st_df["GENDER Focus"] = st_df.apply(focus_group, args=[data, "gender", 1.5], axis=1)
    st_df["URBAN_RURAL Focus"] = st_df.apply(focus_group, args=[data, "urban_rural", 1.45], axis=1)
    st_df[f"{G} Focus"] = st_df.apply(focus_group, args=[data, "geo", 1.3], axis=1)
    st_df["AGE Focus"] = st_df["AGE Focus"].apply(", ".join)
    st_df["URBAN_RURAL Focus"] = st_df["URBAN_RURAL Focus"].apply(", ".join)
    st_df["GENDER Focus"] = st_df["GENDER Focus"].apply(", ".join)
    st_df[f"{G} Focus"] = st_df[f"{G} Focus"].apply(", ".join)

    return st_df

In [ ]:
%%capture

st_df_bar=create_table(data,"barriers")
st_df_bar["Driver/Barrier"]="barrier"

st_df_dri=create_table(data,"drivers")
st_df_dri["Driver/Barrier"]="driver"

merged=pd.concat([st_df_dri,st_df_bar],axis=0,ignore_index=True)


**Note (2026-08-24, confirmed with user):** the `Statements` column is empty for every
`driver` row, by design, not a bug -- `driver_barrier_statement_links.csv`'s
`linked_statements` never links any driver item to a statement, and checking all 62
statement texts in `statement_labels.csv` against all 36 driver reasons confirms why:
the belief statements were all written to test **barrier**-side beliefs (cost, access,
side effects, religion, family approval, fertility-desire), with no driver-side
counterpart. Where a real topical match exists (e.g. desire for more children), it's
coded as a **barrier** item, not a driver, and already has a statement linked.

### Validation

In [ ]:
# ── Validation ────────────────────────────────────────────────────────────────

def _parse_cell(cell_str):
    """Parse a 'subgroup: value[%]' multiline cell into {subgroup: float}.
    Handles both prevalence cells (value%) and count cells (plain number)."""
    result = {}
    if pd.isna(cell_str) or str(cell_str).strip() == "":
        return result
    for line in str(cell_str).split("\n"):
        m = re.match(r"^(.+?):\s*([\d.]+)%?", line.strip())
        if m:
            result[m.group(1).strip()] = float(m.group(2))
    return result


errors = []
warnings_list = []

# 1. Shape
n_drivers  = (merged["Driver/Barrier"] == "driver").sum()
n_barriers = (merged["Driver/Barrier"] == "barrier").sum()
print(f"Rows — drivers: {n_drivers}, barriers: {n_barriers}, total: {len(merged)}")
assert len(merged) > 0, "Output is empty"
assert n_drivers > 0,   "No driver rows found"
assert n_barriers > 0,  "No barrier rows found"

# 2. All expected columns present
missing_cols = [c for c in TABLE_COLUMNS if c not in merged.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
print(f"Columns: all {len(TABLE_COLUMNS)} expected columns present")

# 3. N (All) is non-zero for at least some rows
n_totals = merged["N (All)"].apply(_parse_cell).apply(lambda d: sum(d.values()))
assert n_totals.sum() > 0, "N (All) column is all zeros — count functions may have failed"
print(f"N (All) total across all items: {n_totals.sum():,.0f}")

wn_totals = merged["Weighted N (All)"].apply(_parse_cell).apply(lambda d: sum(d.values()))
print(f"Weighted N (All) total across all items: {wn_totals.sum():,.1f}")

# 4. N ≤ group size for each split
SPLIT_CHECK = [
    ("use",         "N (All)"),
    ("age_group",   "AGE_GROUP: N (All)"),
    ("gender",      "GENDER: N (All)"),
    ("urban_rural", "URBAN_RURAL: N (All)"),
    ("geo",         f"{GEO_SPLIT_LABEL}: N (All)"),
]
for split_var, n_col in SPLIT_CHECK:
    group_sizes = data.groupby(split_var).size().to_dict()
    for _, row in merged.iterrows():
        n_dict = _parse_cell(row[n_col])
        for grp, n_val in n_dict.items():
            max_n = group_sizes.get(grp, None)
            if max_n is not None and n_val > max_n + 0.5:
                errors.append(
                    f"N exceeds group size | {row['Name']} | {split_var}={grp}: "
                    f"N={n_val:.0f} > group size={max_n}"
                )
print(f"N ≤ group-size check: {len(errors)} error(s)")

# 5. Unweighted cross-check: Prevalence (All) ≈ N (All) / group size × 100
TOLERANCE = 3.0  # percentage points
group_sizes = data.groupby("use").size().to_dict()
mismatches = []
for _, row in merged.iterrows():
    prev_dict = _parse_cell(row["Prevalence (All)"])
    n_dict    = _parse_cell(row["N (All)"])
    for grp in set(prev_dict) & set(n_dict):
        if grp not in group_sizes:
            continue
        implied_prev = (n_dict[grp] / group_sizes[grp]) * 100
        diff = abs(prev_dict[grp] - implied_prev)
        if diff > TOLERANCE:
            mismatches.append({
                "item": row["Name"],
                "group": grp,
                "prevalence": prev_dict[grp],
                "implied_from_n": round(implied_prev, 1),
                "diff": round(diff, 1),
            })

if mismatches:
    print(f"\nUnweighted cross-check: {len(mismatches)} mismatch(es) > {TOLERANCE}pp:")
    for m in mismatches[:10]:
        print(f"  {m['item']} | {m['group']}: "
              f"stored={m['prevalence']}%, implied={m['implied_from_n']}%, diff={m['diff']}pp")
else:
    print(f"Unweighted cross-check: Prevalence (All) consistent with N (All) / group size (within {TOLERANCE}pp)")

# 6. No item has counts but missing prevalence (or vice versa)
for _, row in merged.iterrows():
    prev_groups = set(_parse_cell(row["Prevalence (All)"]).keys())
    n_groups    = set(_parse_cell(row["N (All)"]).keys())
    if prev_groups and n_groups and prev_groups != n_groups:
        warnings_list.append(
            f"Subgroup key mismatch | {row['Name']}: "
            f"Prevalence has {prev_groups}, N has {n_groups}"
        )
if warnings_list:
    print(f"\nSubgroup key warnings ({len(warnings_list)}):")
    for w in warnings_list[:5]:
        print(f"  {w}")
else:
    print("Subgroup key alignment: all consistent")

# 7. Flag any rows where N > 0 but Prevalence = 0.0 (rounding artifact)
rounding_flags = []
for _, row in merged.iterrows():
    prev_dict = _parse_cell(row["Prevalence (All)"])
    n_dict    = _parse_cell(row["N (All)"])
    for grp in set(prev_dict) & set(n_dict):
        if n_dict[grp] > 0 and prev_dict[grp] == 0.0:
            rounding_flags.append({"item": row["Name"], "group": grp, "n": n_dict[grp]})
if rounding_flags:
    print(f"\nRounding note — {len(rounding_flags)} case(s) where N > 0 but Prevalence rounds to 0.0%")
    for f in rounding_flags[:5]:
        print(f"  {f['item']} | {f['group']}: N={f['n']}")
else:
    print("Rounding check: no cases of N > 0 with 0.0% Prevalence")

# Summary
if errors:
    print(f"\n{'='*60}\n{len(errors)} VALIDATION ERROR(S) — review before uploading\n{'='*60}")
    for e in errors:
        print(f"  ERROR: {e}")
else:
    print("\nAll validation checks passed.")

In [ ]:

merged.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

In [ ]:
def main_question_freq(data, col, split_var=None):
    """
    Frequency table for a select_one column.
    Returns a DataFrame with unweighted n, unweighted %, and weighted % per answer.
    If split_var is provided, adds columns for each subgroup's weighted %.
    """
    df = data[[col, "normalized_weight"]].dropna(subset=[col]).copy()

    base = (
        df.groupby(col)
        .agg(n=(col, "count"), weighted_n=("normalized_weight", "sum"))
        .assign(
            unweighted_pct=lambda x: round(x["n"] / x["n"].sum() * 100, 1),
            weighted_pct=lambda x: round(x["weighted_n"] / x["weighted_n"].sum() * 100, 1),
        )
        .sort_values("weighted_pct", ascending=False)
        .drop(columns="weighted_n")
    )

    if split_var:
        for grp, grp_df in data.groupby(split_var):
            grp_data = grp_df[[col, "normalized_weight"]].dropna(subset=[col])
            split_wgt = (
                grp_data.groupby(col)["normalized_weight"].sum()
                / grp_data["normalized_weight"].sum() * 100
            ).round(1).rename(f"{grp} weighted %")
            base = base.join(split_wgt, how="left")

    return base.fillna(0)


print("=" * 60)
print("MAIN BARRIER (select_one) — weighted frequency")
print("=" * 60)
display(main_question_freq(data, "main_barriers", split_var="use"))

print("\n" + "=" * 60)
print("MAIN DRIVER (select_one) — weighted frequency")
print("=" * 60)
display(main_question_freq(data, "main_drivers", split_var="use"))